# 💳 Credit Scoring Model — EDA & Modeling

**Objective:** Predict individual creditworthiness using past financial data.

**Approach:** Train and compare Logistic Regression, Decision Tree, and Random Forest classifiers.

**Metrics:** Precision, Recall, F1-Score, ROC-AUC

In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✅')

## 1. Generate & Load Data

In [ ]:
from generate_data import generate_credit_data

df = generate_credit_data(n_samples=2000, seed=42)
df.to_csv('../data/credit_data.csv', index=False)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(5, 3.5))
counts = df['creditworthy'].value_counts()
ax.bar(['Not Creditworthy (0)', 'Creditworthy (1)'], counts.values,
       color=['#E07070', '#70B870'], edgecolor='white')
ax.set_title('Target Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by class
num_features = ['income', 'payment_history', 'credit_utilization',
                'debt_to_income_ratio', 'num_late_payments', 'employment_years']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, num_features):
    for label, color in [(0, '#E07070'), (1, '#70B870')]:
        df[df['creditworthy'] == label][col].hist(
            bins=30, ax=ax, alpha=0.6, color=color,
            label=f'Creditworthy={label}'
        )
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.legend(fontsize=8)
fig.suptitle('Feature Distributions by Creditworthiness', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
from preprocess import load_and_preprocess

X_train, X_test, y_train, y_test, feature_names, scaler = load_and_preprocess()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features ({len(feature_names)}): {feature_names}')

## 4. Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
}

for name, clf in classifiers.items():
    cv = cross_val_score(clf, X_train, y_train, cv=5, scoring='roc_auc')
    clf.fit(X_train, y_train)
    print(f'{name:25s} → CV ROC-AUC: {cv.mean():.4f} ± {cv.std():.4f}')

## 5. Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

results = {}
for name, clf in classifiers.items():
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    results[name] = {'y_pred': y_pred, 'y_prob': y_prob}
    print(f'\n{"-"*50}\n{name}\n{"-"*50}')
    print(classification_report(y_test, y_pred, target_names=['Not Creditworthy', 'Creditworthy']))

In [ ]:
# ROC Curves
colors = ['#4C72B0', '#DD8452', '#55A868']
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], 'k--', lw=1.2)

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc = roc_auc_score(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
rf = classifiers['Random Forest']
importance = pd.Series(rf.feature_importances_, index=feature_names).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot.barh(ax=ax, color='#55A868', alpha=0.85)
ax.set_title('Random Forest — Feature Importances', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 6. Summary Table

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rows = []
for name, res in results.items():
    rows.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, res['y_pred']), 4),
        'Precision': round(precision_score(y_test, res['y_pred']), 4),
        'Recall':    round(recall_score(y_test, res['y_pred']), 4),
        'F1-Score':  round(f1_score(y_test, res['y_pred']), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, res['y_prob']), 4),
    })

summary = pd.DataFrame(rows).set_index('Model')
summary.style.highlight_max(axis=0, color='#c6efce')